[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# Del *notebook* al informe y cierre del curso.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo IV: *Del análisis al informe*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se exportará un análisis a informe con `nbconvert`, se practicará la redacción y la revisión entre pares, y se cerrará el curso con un repaso.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

* Markdown en *Jupyter*, para redactar el informe dentro del *notebook*.
* Exportación: [documentación de `nbconvert`](https://nbconvert.readthedocs.io/en/latest/usage.html).
* Gráficas: [py311 · 32 Introducción a *Seaborn*](https://github.com/PythonistaMX/py311/blob/main/32_introduccion_a_seaborn.ipynb).

> ⚠️ **Aviso:** este curso usa *Pandas* 3. El código escrito para *Pandas* 2 o versiones anteriores puede dar resultados distintos o lanzar errores. Si necesitas repasar *Pandas*, consulta el curso [py311](https://github.com/PythonistaMX/py311).

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, `stats` de *SciPy* y los módulos de `nbformat` y `nbconvert` para la exportación, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
import tempfile

import nbformat
from nbconvert import HTMLExporter
from scipy import stats

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará desde `penguins.csv` si no existe) y `mpg.csv`, y creará `litros_100km`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
autos = pd.read_csv(os.path.join(DATA, "mpg.csv")).assign(litros_100km=lambda d: 235.215 / d["mpg"])
pinguinos.shape, autos.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `contar_palabras` y el resumen ejecutivo `resumen_mpg` de `25_estructura_del_informe_y_comunicacion.ipynb`.

In [ ]:
def contar_palabras(texto):
    return len(texto.split())


resumen_mpg = (
    "Se analizaron 398 modelos de automóviles vendidos en EE. UU. entre 1970 y 1982 para saber qué "
    "características se asocian con el consumo de combustible y si este difiere según el origen. El consumo "
    "se asocia sobre todo con el tamaño del auto: peso, motor, potencia y cilindros miden casi lo mismo, y el "
    "peso por sí solo se relaciona de forma casi lineal con los litros por cada 100 km (r = 0.89). Los modelos "
    "de EE. UU. consumen unos 4 a 5 L/100 km más que los europeos y japoneses, pero también pesan entre 940 y 1,140 lb "
    "más; entre autos ligeros de peso parecido, la diferencia (en la mitad más ligera) baja a menos de 1 L/100 km. Principal limitación: "
    "cada fila es un modelo, no un auto vendido, y no hay autos pesados europeos o japoneses con los cuales "
    "comparar."
)
contar_palabras(resumen_mpg)

---
## 1. Del *notebook* al informe: `nbconvert`.

`nbconvert` transforma un `.ipynb` en otros formatos. Desde la terminal del *Codespace*:

```
jupyter nbconvert --to html --no-input <notebook>.ipynb
jupyter nbconvert --to webpdf --no-input <notebook>.ipynb
```

Donde:
* `--to html` genera un HTML autocontenido (con las gráficas incrustadas) que se abre en cualquier navegador.
* `--to webpdf` genera un PDF a partir del HTML. Requiere instalar una vez `pip install "nbconvert[webpdf]"` y `playwright install chromium`. `--to pdf` usa LaTeX, que no viene en el *Codespace*.
* `--no-input` oculta las celdas de código y deja solo el Markdown y las salidas: es lo que se comparte como **informe**. Sin esa opción se exporta el análisis completo.
* `--execute` vuelve a ejecutar el *notebook* antes de exportar: una forma rápida de verificar la reproducibilidad.

**Ejemplo:** la misma conversión se puede hacer desde *Python* con `HTMLExporter`. Se armará un *notebook* mínimo con el resumen y una tabla, y se exportará a una carpeta temporal para no dejar archivos en el repositorio.

* La siguiente celda creará un *notebook* mínimo con `nbformat`, lo exportará a HTML con y sin código, y comparará el tamaño de ambos y si el código aparece en cada uno.

In [ ]:
informe_demo = nbformat.v4.new_notebook()
informe_demo.cells = [
    nbformat.v4.new_markdown_cell("# Informe de prueba.\n\n" + resumen_mpg),
    nbformat.v4.new_code_cell("print('consumo mediano por origen (L/100 km): usa 12.7, europe 8.9, japan 7.4')"),
]
informe_demo.cells[1].outputs = [nbformat.v4.new_output("stream", name="stdout",
                                 text="consumo mediano por origen (L/100 km): usa 12.7, europe 8.9, japan 7.4\n")]
resultados_exportacion = {}
with tempfile.TemporaryDirectory() as carpeta:
    for ocultar_codigo in [False, True]:
        exportador = HTMLExporter(exclude_input=ocultar_codigo)
        html, _ = exportador.from_notebook_node(informe_demo)
        ruta_html = os.path.join(carpeta, f"informe_sin_codigo_{ocultar_codigo}.html")
        with open(ruta_html, "w", encoding="utf-8") as archivo:
            archivo.write(html)
        resultados_exportacion["sin código" if ocultar_codigo else "con código"] = {
            "kB": round(os.path.getsize(ruta_html) / 1024, 1),
            "¿aparece el código?": ">print</span>" in html,
            "¿aparece la salida?": "usa 12.7" in html,
        }
pd.DataFrame(resultados_exportacion).T

**Interpretación:** las dos versiones contienen el texto y la salida; solo la versión **con código** contiene la instrucción `print` (en el HTML el código aparece resaltado con etiquetas `<span>`, por eso se busca `>print</span>`). Ambos HTML pesan cientos de kB porque incluyen los estilos de *JupyterLab*. **Decisión que permite tomar:** se comparte el HTML **sin código** como informe y el `.ipynb` ejecutado como análisis.

**Errores comunes que se evitan:**
* Exportar un *notebook* que no se ejecutó completo: el HTML muestra salidas viejas o vacías. Siempre **Restart & Run All** antes de exportar.
* Celdas de Markdown que solo tienen sentido junto al código ("la siguiente celda…"): en el informe sin código se leen raras. Redacta el informe en una sección final propia.

**💡 Ideas clave de la sección 1:**
* `jupyter nbconvert --to html --no-input` produce el informe; `--to webpdf` lo convierte a PDF.
* Se exporta solo después de **Restart & Run All**.

---
## 2. Taller de redacción y revisión entre pares.

**Lista de verificación para revisar un informe (el propio o el de un compañero, por ejemplo los textos de los ejercicios 1 y 2):**

| # | Pregunta | Sí / No |
|---|---|---|
| 1 | ¿El resumen ejecutivo tiene 150 palabras o menos y dice **qué se encontró**, con números? | |
| 2 | ¿Se describe de dónde vienen los datos y a qué población representan? | |
| 3 | ¿Cada decisión de limpieza dice cuántas filas afecta? | |
| 4 | ¿Cada hallazgo tiene estimación, IC y tamaño del efecto? | |
| 5 | ¿Se revisaron los supuestos de cada prueba? | |
| 6 | ¿Las gráficas finales tienen título con el hallazgo, unidades y *n*? | |
| 7 | ¿Se evitan las frases causales y las generalizaciones injustificadas? | |
| 8 | ¿Las limitaciones mencionan sesgos y confusores concretos? | |
| 9 | ¿El *notebook* corre con **Restart & Run All**? | |

**Cómo dar retroalimentación:** una fortaleza concreta, los dos cambios más importantes (no veinte) y una pregunta que el informe no responde.

### Ejercicio 1 · Resumen ejecutivo de una comparación.

**🎯 Objetivo:** redactar la pieza más leída del informe. **Qué se espera aprender:** a condensar el análisis en una respuesta con números, incertidumbre y límites, en máximo 150 palabras.

Con los resultados de la sección 1 de `20_pruebas_para_dos_grupos.ipynb`, escribe en `resumen_pinguinos` el resumen ejecutivo de la comparación de masa corporal entre machos y hembras Adelie y verifica su extensión con `contar_palabras`. Sigue la misma estructura que `resumen_mpg`: qué se analizó, la respuesta con su número e incertidumbre y la principal limitación.

* La siguiente celda muestra una solución posible.

In [ ]:
resumen_pinguinos = (
    "Se compararon las masas corporales de 73 machos y 73 hembras Adelie de tres islas del archipiélago Palmer. "
    "Los machos pesan en promedio unos 675 g más que las hembras (IC 95%: 573 a 776 g) y, al comparar un macho "
    "y una hembra al azar, el macho resulta más pesado en unos 94 de cada 100 casos. Una diferencia así de grande, "
    "con un intervalo tan lejos de cero, sería muy rara solo por la variación muestral. Principal limitación: "
    "son aves de una sola especie y de tres islas, que no se eligieron al azar; no se puede generalizar a otras "
    "poblaciones."
)
contar_palabras(resumen_pinguinos)

**✅ Respuesta sugerida:**

Ejemplo con la comparación de masa entre machos y hembras Adelie (103 palabras): sigue la misma estructura que el de `mpg`, es decir, qué se analizó, la respuesta con su número y su IC, la probabilidad de superioridad y la limitación principal.

### Ejercicio 2 · Un hallazgo completo.

**🎯 Objetivo:** producir un hallazgo con todas sus piezas. **Qué se espera aprender:** a encadenar estimación, IC, tamaño del efecto, gráfica final e interpretación en lenguaje llano.

Con `autos`, redacta el hallazgo "los modelos de EE. UU. consumen más que los japoneses": calcula la diferencia de **medianas** de `litros_100km` con su IC *bootstrap* percentil del 95% (`random_state=401`), la probabilidad de superioridad y una gráfica final que cumpla la lista de la sección 3.2 de `25_estructura_del_informe_y_comunicacion.ipynb`. Termina con una frase en lenguaje llano que mencione el confusor.

* La siguiente celda muestra una solución posible.

In [ ]:
consumo_usa = autos.loc[autos["origin"] == "usa", "litros_100km"].to_numpy()
consumo_japon = autos.loc[autos["origin"] == "japan", "litros_100km"].to_numpy()


def diferencia_medianas(a, b, axis=-1):
    return np.median(a, axis=axis) - np.median(b, axis=axis)


ic_origen = stats.bootstrap((consumo_usa, consumo_japon), diferencia_medianas, n_resamples=9999,
                            method="percentile", random_state=401).confidence_interval
superioridad = stats.mannwhitneyu(consumo_usa, consumo_japon).statistic / (len(consumo_usa) * len(consumo_japon))

dos_origenes = autos[autos["origin"].isin(["usa", "japan"])].assign(
    origen=lambda d: d["origin"].map({"usa": f"EE. UU. (n = {len(consumo_usa)})", "japan": f"Japón (n = {len(consumo_japon)})"}))
fig, ax = plt.subplots(figsize=(7, 4))
sns.stripplot(dos_origenes, x="origen", y="litros_100km", alpha=0.35, color="gray", ax=ax)
sns.pointplot(dos_origenes, x="origen", y="litros_100km", estimator="median", errorbar=("pi", 50), color="black",
              linestyle="none", markers="D", capsize=0.1, ax=ax)
ax.set(xlabel="", ylabel="consumo (L/100 km)",
       title="Los modelos de EE. UU. consumen unos 5 L/100 km más que los japoneses\n(rombos: mediana y rango intercuartílico)")
sns.despine()
plt.show()
{"diferencia de medianas": round(float(diferencia_medianas(consumo_usa, consumo_japon)), 2),
 "IC 95%": (round(float(ic_origen.low), 2), round(float(ic_origen.high), 2)),
 "probabilidad de superioridad": round(float(superioridad), 3)}

**✅ Respuesta sugerida:**

La mediana del consumo de los modelos de EE. UU. es unos 5.3 L/100 km mayor que la de los japoneses (IC 95% *bootstrap*: alrededor de 4.4 a 5.7 L/100 km), y en cerca de 87 de cada 100 pares EE. UU.-Japón el modelo estadounidense consume más. **En lenguaje llano:** "Los modelos de EE. UU. de 1970 a 1982 consumían bastante más combustible que los japoneses, pero también eran mucho más pesados; entre autos ligeros de peso parecido la diferencia es menor de 1 L/100 km y no hay modelos japoneses pesados con los que comparar (`23_eda_completo_con_mpg.ipynb`), así que buena parte de la brecha va junto con el tamaño de los autos."

### Ejercicio 3 · Corrige las frases.

**🎯 Objetivo:** detectar afirmaciones que los datos no respaldan. **Qué se espera aprender:** a revisar un informe (propio o ajeno) con la tabla de la sección 3.3 de `25_estructura_del_informe_y_comunicacion.ipynb`.

Reescribe cada frase para que sea correcta:
1. "El origen japonés hace que los autos consuman menos."
2. "La prueba de Welch demostró con 99.9% de certeza que los machos Adelie pesan más."
3. "Como *p* = 0.59, los Chinstrap y los Adelie pesan lo mismo."
4. "La correlación de 0.89 indica que 89% del consumo se debe al peso."

**✅ Respuesta sugerida:**

1. "Los modelos japoneses consumían menos en promedio, pero también eran más ligeros; entre autos ligeros de peso parecido la diferencia casi desaparece." (Lenguaje causal y confusor omitido.)
2. "Los machos Adelie pesaron en promedio unos 675 g más (IC 95%: 573 a 776 g); una diferencia así sería muy improbable si los sexos no difirieran (*p* < 0.001)." (El p-value no es una certeza.)
3. "No se encontró evidencia de diferencia de masa entre Chinstrap y Adelie; los datos son compatibles con diferencias de −86 a 150 g." (No rechazar H0 no demuestra igualdad.)
4. "Los autos más pesados tienden a consumir más; la relación es fuerte (*r* = 0.89)." Si se quiere una proporción, *r* al cuadrado (≈ 0.78) es la fracción de la **variabilidad** del consumo asociada linealmente con el peso, no la fracción del consumo "causada" por el peso.

### Reto 1 · Exporta tu propio informe.

**🎯 Objetivo:** practicar una exportación real a HTML. **Qué se espera aprender:** a detectar, en el HTML, lo que se lee mal sin el código.

En la terminal del *Codespace*, exporta `23_eda_completo_con_mpg.ipynb` a HTML **sin código** con `jupyter nbconvert --to html --no-input --execute --output-dir salidas 23_eda_completo_con_mpg.ipynb` (la carpeta `salidas/` la ignora *git*). Ábrelo en el navegador y anota tres cosas que se leen mal sin el código (frases como "la siguiente celda…", tablas sin título, gráficas sin unidades). Escribe cómo las corregirías para que el informe se entienda solo.

**✅ Respuesta sugerida:**

Problemas típicos: celdas de Markdown que remiten al código (en el `23`, cada celda de código va precedida de "La siguiente celda…"), salidas largas de `describe()` sin una frase que diga qué mirar, gráficas exploratorias que no se eliminaron y títulos genéricos. La solución es redactar el informe en una sección propia, que se pueda leer sola, con sus propias tablas resumidas y gráficas finales (como la sección 2.6 del `23`, "Conclusiones de la demostración").

### Reto 2 · Probabilidad de superioridad e incertidumbre.

**🎯 Objetivo:** acompañar el tamaño del efecto con su propia incertidumbre. **Qué se espera aprender:** que cualquier estadístico, incluso uno "para el público", tiene un IC.

Calcula un IC *bootstrap* percentil del 95% (`random_state=401`) para la probabilidad de superioridad de los machos sobre las hembras **Adelie** en masa corporal. Redacta la frase en lenguaje llano.

* La siguiente celda muestra una solución posible.

In [ ]:
adelie = pinguinos[pinguinos["species"] == "Adelie"]
machos_adelie = adelie.loc[adelie["sex"] == "Male", "body_mass_g"].to_numpy()
hembras_adelie = adelie.loc[adelie["sex"] == "Female", "body_mass_g"].to_numpy()


def prob_superioridad(a, b, axis=-1):
    a, b = np.moveaxis(a, axis, -1), np.moveaxis(b, axis, -1)
    mayor = (a[..., :, None] > b[..., None, :]).mean(axis=(-2, -1))
    empate = (a[..., :, None] == b[..., None, :]).mean(axis=(-2, -1))
    return mayor + 0.5 * empate


ic_ps = stats.bootstrap((machos_adelie, hembras_adelie), prob_superioridad, n_resamples=1999, method="percentile",
                        random_state=401, batch=200).confidence_interval
round(float(prob_superioridad(machos_adelie, hembras_adelie)), 3), (round(float(ic_ps.low), 3), round(float(ic_ps.high), 3))

**✅ Respuesta sugerida:**

La probabilidad de superioridad es 0.942 (IC 95% *bootstrap* aproximadamente de 0.91 a 0.97). **En lenguaje llano:** "Al comparar un macho y una hembra Adelie elegidos al azar, el macho pesa más en unos 94 de cada 100 casos (entre 91 y 97, con 95% de confianza)." Se usan 1,999 remuestras y `batch=200` para limitar la memoria, porque cada remuestra compara todos los pares.

---
## 3. Repaso y cierre del curso.

### 3.1 📝 Preguntas de repaso.

**Diez preguntas para repasar los módulos III y IV** (*notebooks* `16_distribucion_muestral` a `26_del_notebook_al_informe`). Elige una opción en cada una y justifícala en una frase:
1. Un IC del 95% para la media significa que: a) hay 95% de probabilidad de que la media muestral esté en el intervalo; b) el procedimiento captura la media poblacional en 95% de las muestras repetidas; c) el 95% de los datos está en el intervalo; d) la hipótesis nula es falsa con 95% de certeza.
2. Al aumentar el tamaño de muestra de *n* a 4*n*, el error estándar de la media: a) se duplica; b) se reduce a la mitad; c) se reduce a la cuarta parte; d) no cambia.
3. El *bootstrap* estima la distribución muestral: a) suponiendo normalidad; b) remuestreando con reemplazo los datos observados; c) remuestreando sin reemplazo; d) recolectando datos nuevos.
4. Un p-value de 0.03 significa: a) que H0 tiene 3% de probabilidad de ser cierta; b) que, si H0 fuera cierta, un resultado al menos así de extremo ocurriría alrededor de 3% de las veces; c) que el efecto es grande; d) que el estudio tiene 97% de potencia.
5. Para comparar las medias de dos grupos independientes con varianzas distintas conviene: a) *t* pareada; b) *t* de Welch; c) chi cuadrada; d) correlación de Pearson.
6. Para probar si la clase del pasajero y la supervivencia son independientes se usa: a) ANOVA; b) *t* de Student; c) chi cuadrada de independencia; d) regresión lineal.
7. Un supuesto clave de la chi cuadrada es: a) que las frecuencias esperadas sean suficientemente grandes (≥ 5 en general); b) normalidad de los datos; c) igualdad de medias; d) datos pareados.
8. Un ANOVA con *p* < 0.05 entre 4 grupos indica: a) que los 4 grupos son distintos entre sí; b) que al menos un grupo difiere y se requiere una prueba *post hoc* (por ejemplo, Tukey) para saber cuáles; c) que el efecto es grande; d) que hay causalidad.
9. Con *n* = 100,000 por grupo, una diferencia de medias tiene *p* < 0.001 y d de Cohen = 0.03. Lo correcto es concluir que: a) es un hallazgo muy importante; b) es estadísticamente significativa pero de magnitud muy pequeña (si importa depende del contexto); c) es un error; d) hay que rechazar el resultado.
10. En un informe de hallazgos, un estudio observacional permite afirmar: a) "X causa Y"; b) "X se asocia con Y", señalando posibles confusores; c) "X no influye en Y" si *p* > 0.05; d) nada.

**✅ Respuestas:**

1. **b.** El 95% describe el procedimiento: de muchas muestras repetidas, alrededor del 95% de los intervalos captura la media de la población, que es un valor fijo y no tiene "probabilidad" (`17_error_estandar_e_intervalos_de_confianza.ipynb`).
2. **b.** El error estándar es la desviación estándar dividida entre la raíz cuadrada de *n*; con 4*n* la raíz se duplica y el error se reduce a la mitad (`16_distribucion_muestral.ipynb`).
3. **b.** El *bootstrap* imita nuevas muestras remuestreando con reemplazo los datos observados, sin suponer normalidad (`18_bootstrap_e_interpretacion_de_intervalos.ipynb`).
4. **b.** El p-value es la probabilidad de un resultado así de extremo **si H0 fuera cierta**; no es la probabilidad de que H0 sea cierta (`19_logica_del_contraste.ipynb`).
5. **b.** La *t* de Welch no exige varianzas iguales; la pareada es para mediciones del mismo individuo, la chi cuadrada para conteos y Pearson para dos variables numéricas (`20_pruebas_para_dos_grupos.ipynb`).
6. **c.** Clase y supervivencia son dos variables categóricas, y la chi cuadrada de independencia compara conteos (`21_proporciones_y_mas_de_dos_grupos.ipynb`).
7. **a.** La chi cuadrada es una aproximación que necesita frecuencias esperadas suficientemente grandes (≥ 5 en general); no pide normalidad de los datos (`21_proporciones_y_mas_de_dos_grupos.ipynb`).
8. **b.** El ANOVA solo dice que al menos un grupo difiere; para saber cuáles se necesita una prueba *post hoc* como Tukey (`21_proporciones_y_mas_de_dos_grupos.ipynb`).
9. **b.** Con una muestra enorme hasta una diferencia diminuta sale significativa; una d de 0.03 es muy pequeña y el contexto decide si importa (`22_tamano_del_efecto.ipynb`).
10. **b.** Un estudio observacional permite hablar de asociación, no de causa, y señalar los posibles confusores; un *p* > 0.05 no demuestra que no haya efecto (`15_visualizacion_efectiva.ipynb` y `25_estructura_del_informe_y_comunicacion.ipynb`).

### 3.2 ✅ Cierre del curso.

**En el curso vimos:**
* **Módulo I:** *NumPy* y *pandas* 3, tipos de variables y escalas de medición, muestreo y sesgos, calidad de datos y faltantes.
* **Módulo II:** distribuciones, medidas robustas y atípicos; agregación, relaciones, confusores y visualización.
* **Módulo III:** distribución muestral, intervalos de confianza y *bootstrap*; pruebas de hipótesis y tamaño del efecto.
* **Módulo IV:** un EDA completo y reproducible, y su informe de hallazgos.

**Hoy vimos:** la estructura del informe, cómo traducir IC y tamaños del efecto a lenguaje llano, la gráfica final, qué no afirmar, la exportación con `nbconvert` y la revisión entre pares.

**Siguiente paso:** repetir el flujo de `23_eda_completo_con_mpg.ipynb` con el *dataset* que más te interese, con las mismas funciones y la lista de verificación de este *notebook*. ¡Gracias por participar!

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>